# 01 · เตรียมข้อมูล

อ่านไฟล์คำตอบแบบสอบถาม แปลงคำตอบเป็นตัวเลข คิดคะแนน DASS-21 แล้วเขียน `data/data_clean.csv`
ที่ notebook อื่นทุกไฟล์อ่าน จึงต้องรันไฟล์นี้ก่อนหนึ่งครั้ง

## 1.1 ตั้งค่า

ค่าตั้งที่ทุก notebook ใช้ร่วมกันอยู่ใน `common.py` ได้แก่ พาธของข้อมูลและรูป ค่าคงที่
(seed = 42 ระดับนัยสำคัญ 0.05 และ bootstrap 5,000 รอบ) ฟอนต์ภาษาไทยของกราฟ เลขข้อของ DASS-21 แต่ละมิติ
และฟังก์ชันอ่านกลุ่มตัวอย่าง ทุกไฟล์จึงขึ้นต้นด้วยสองเซลล์เดียวกัน คือเซลล์เปลี่ยนโฟลเดอร์ และเซลล์ import

In [1]:
# ไลบรารีทั้งหมดที่ notebook ทุกไฟล์ใช้ ทั้ง `uv sync` และ Colab มีครบอยู่แล้ว
# ถ้าเครื่องยังไม่มี ให้ลบ # หน้าบรรทัดล่างแล้วรันเซลล์นี้ก่อน
# %pip install numpy pandas scipy statsmodels matplotlib pillow

In [2]:
import os
import pathlib
import sys

# notebook.nvim starts the kernel in nvim's cwd; step into codingpy/ so common.py
# and data/ are found when the notebook is opened from the repo root
if pathlib.Path("codingpy/common.py").is_file():
    os.chdir("codingpy")
    sys.path.insert(0, os.getcwd())

%matplotlib inline

In [3]:
import platform
import re

import matplotlib
import numpy as np
import pandas as pd
import scipy
import statsmodels
from IPython.display import display

from common import (
    CLEAN_PATH,
    CODEBOOK_PATH,
    DASS_ITEMS,
    DIMENSIONS,
    IN_COLAB,
    RAW_PATH,
    SPSS_PATH,
    SPSS_SYNTAX_PATH,
    THAI_FONT,
    load_data,
    passed_attention,
)

print(f"Python {platform.python_version()}")
for module in (np, pd, scipy, statsmodels, matplotlib):
    print(f"{module.__name__} {module.__version__}")
print(f"Thai font: {THAI_FONT}")

Python 3.13.14
numpy 2.5.1
pandas 3.0.5
scipy 1.18.0
statsmodels 0.15.0
matplotlib 3.11.1
Thai font: Sarabun


## 1.2 นำเข้าข้อมูล

ไฟล์ `data/data.csv` คือคำตอบที่ส่งออกจาก Google Forms หนึ่งแถวต่อผู้ตอบหนึ่งคน
หัวคอลัมน์ที่ส่งออกมาสะกดผิด 3 ชื่อ (`main_platfrom`, `sleep_hour`, `exersice_days`) จึงแก้ตั้งแต่ตอนอ่านไฟล์

บน Colab ถ้ายังไม่มีไฟล์ เซลล์นี้จะเปิดช่องให้อัปโหลด

In [4]:
# the export misspells three headers; fix them before anything else reads them
RENAME = {
    "main_platfrom": "main_platform",
    "sleep_hour": "sleep_hours",
    "exersice_days": "exercise_days",
}

if IN_COLAB and not RAW_PATH.exists():
    from google.colab import files

    # pick data.csv in the dialog; it is kept as data/data.csv for this session
    uploaded = files.upload()
    RAW_PATH.write_bytes(next(iter(uploaded.values())))

if not RAW_PATH.exists():
    raise FileNotFoundError(f"put the survey export at {RAW_PATH} before running")

raw = pd.read_csv(RAW_PATH).rename(columns=RENAME)
print(f"{raw.shape[0]} responses x {raw.shape[1]} columns from {RAW_PATH}")

121 responses x 39 columns from data/data.csv


ตารางด้านล่างสรุปคอลัมน์ทั้งหมด หนึ่งบรรทัดต่อหนึ่งคอลัมน์ ไม่ใช่ต่อผู้ตอบ
คอลัมน์ตัวเลขอย่าง `sleep_hours` หรือ `GPA` ยังเป็นข้อความ เพราะผู้ตอบพิมพ์เองได้อิสระ เช่น "6-7 ชม."
ข้อ 1 ถึง 21 คือ DASS-21 และคอลัมน์ที่ขึ้นต้นว่า "ข้อนี้ไม่ใช่คำถาม" คือข้อดักความใส่ใจ

In [5]:
# one line per column, not per respondent: the answers themselves stay out of the notebook
overview = pd.DataFrame(
    {
        "column": [str(c).strip()[:48] for c in raw.columns],
        "answers": raw.notna().sum().to_numpy(),
        "distinct": raw.nunique().to_numpy(),
        "dtype": raw.dtypes.astype(str).to_numpy(),
    }
)
display(overview)

,column,answers,distinct,dtype
0,ประทับเวลา,121,121,str
1,ข้อมูลที่สอบถาม ได้แก่ อายุ เพศ ชั้นปี คณะ/สาขา,121,1,str
2,age,121,6,int64
3,gender,121,2,str
4,year_of_study,121,3,str
5,faculty,87,67,str
6,social_media_hours,121,56,str
7,main_platform,121,5,str
8,content_type,121,9,str
9,news_days,121,7,int64


## 1.3 แปลงคำตอบภาษาไทยเป็นป้ายภาษาอังกฤษ

คำตอบแบบตัวเลือกถูกเก็บเป็นข้อความภาษาไทย การวิเคราะห์และ SPSS ต้องการป้ายที่คงที่ จึงรวมตารางแปลงไว้ที่เดียว

- **เพศ แพลตฟอร์ม ประเภทเนื้อหา** แปลงตรงตัวด้วย dict
- **กลุ่มเนื้อหา** ยุบ 9 ประเภทเหลือ 4 กลุ่มตามที่เค้าโครงกำหนด (ข่าว บันเทิง ชีวิตประจำวันของคนอื่น และอื่น ๆ)
  เพราะบางประเภทมีผู้ตอบน้อยเกินกว่าจะเทียบกลุ่มได้
- **คณะ** เป็นข้อความที่พิมพ์เอง มีการสะกด 66 แบบสำหรับราว 13 คณะ จึงจับด้วยคำสำคัญตามลำดับ
  คำสำคัญของคณะมาก่อนคำสำคัญของสาขา ("คณะวิทยาศาสตร์ สาขาวิทยาการคอมพิวเตอร์" จึงนับเป็นวิทยาศาสตร์)
- **รหัส SPSS** คือลำดับของป้ายใน `CODE_ORDER` เริ่มจาก 1

In [6]:
GENDER = {
    "ชาย": "Male",
    "หญิง": "Female",
    "อื่น ๆ": "Other",
    "อื่นๆ": "Other",
    "ไม่ประสงค์ระบุ": "Prefer not to say",
}

PLATFORM = {
    "Tiktok": "TikTok",
    "Instagram": "Instagram",
    "Youtube": "YouTube",
    "Facebook": "Facebook",
    "X": "X",
}

CONTENT_TYPE = {
    "ข่าวสารบ้านเมือง การเมือง เหตุการณ์ปัจจุบัน": "News and politics",
    "บันเทิง (ซีรีส์ อนิเมะ คลิปตลก เพลง)": "Entertainment",
    "เกม / อีสปอร์ต": "Games and esports",
    "กีฬา": "Sports",
    "ความรู้ การเรียน พัฒนาตนเอง": "Education and self-improvement",
    "ชีวิตประจำวันของคนอื่น (เพื่อน คนดัง อินฟลูเอนเซอร์)": "Other people's daily life",
    "สินค้า รีวิว ช้อปปิ้ง": "Products and shopping",
    "Podcast": "Podcast",
    "การทำอาหาร": "Cooking",
}

# the four-group collapse fixed in the proposal, used when a cell is too small
# for a group comparison (News 11, Other people's life 15, Entertainment 61)
CONTENT_GROUP = {
    "News and politics": "News",
    "Entertainment": "Entertainment",
    "Other people's daily life": "Others' daily life",
}
CONTENT_GROUP_FALLBACK = "Other"

# faculty is free text: 66 spellings for ~13 faculties. Rules are tried in
# order, so a faculty keyword wins over the field-of-study keyword that follows
# it ("คณะวิทยาศาสตร์ สาขาวิทยาการคอมพิวเตอร์" is Science, not Computing).
FACULTY_RULES = [
    ("Education", ["ครุศาสตร์", "ศึกษาศาสตร์"]),
    ("Humanities and Social Sciences", ["มนุษย", "มนุษฯ", "สังคมศาสตร์"]),
    ("Law", ["นิติ"]),
    (
        "Political Science and Public Affairs",
        ["รัฐศาสตร์", "รปศ", "นโยบายสาธารณะ", "COPA"],
    ),
    ("Agriculture", ["เกษตร"]),
    ("Engineering", ["วิศวกรรม", "EN"]),
    ("Architecture and Fine Arts", ["สถาปัตย", "ศิลปกรรม"]),
    ("Public Health", ["สาธารณสุข"]),
    ("International College", ["นานาชาติ"]),
    (
        "Science",
        ["วิทยาศาสตร์", "วืทยาศาสตร์", "ฟิสิกส์", "สถิติ", "วิทยาการคำนวณ"],
    ),
    (
        "Business and Management",
        ["บริหาร", "การตลาด", "วิทยาการจัดการ", "ธุรกิจ", "KKBS"],
    ),
    (
        "College of Computing",
        ["คอม", "วิทคอม", "ปัญญาประดิษฐ์", "CP", "CS", "GIS", "AI"],
    ),
    ("Communication Arts", ["นิเทศ"]),
    ("Interdisciplinary Studies", ["สหวิทยาการ"]),
]
FACULTY_FALLBACK = "Other"

# order = SPSS code order; the code is the position in the list, starting at 1
CODE_ORDER = {
    "gender": ["Male", "Female", "Other", "Prefer not to say"],
    "faculty_group": [name for name, _ in FACULTY_RULES] + [FACULTY_FALLBACK],
    "main_platform": ["TikTok", "Instagram", "YouTube", "Facebook", "X"],
    "content_type": list(CONTENT_TYPE.values()),
    "content_group": [
        "News",
        "Entertainment",
        "Others' daily life",
        CONTENT_GROUP_FALLBACK,
    ],
}


def faculty_group(value) -> float | str:
    """Bucket a free-text faculty answer into one English faculty name."""
    if pd.isna(value) or not str(value).strip():
        return np.nan
    text = str(value)
    upper = text.upper()
    for name, keywords in FACULTY_RULES:
        for kw in keywords:
            # ASCII abbreviations (CP, CS, EN) must match as whole words so that
            # "CS" does not fire inside an unrelated English word
            if re.fullmatch(r"[A-Za-z]+", kw):
                if re.search(rf"\b{kw}\b", upper):
                    return name
            elif kw in text:
                return name
    return FACULTY_FALLBACK


def to_english(df: pd.DataFrame) -> pd.DataFrame:
    """Replace every Thai categorical answer with its English label."""
    out = df.copy()
    out["gender"] = out["gender"].map(GENDER).fillna(out["gender"])
    out["main_platform"] = (
        out["main_platform"].map(PLATFORM).fillna(out["main_platform"])
    )
    out["content_type"] = (
        out["content_type"].map(CONTENT_TYPE).fillna(out["content_type"])
    )
    out["content_group"] = out["content_type"].map(
        lambda v: np.nan if pd.isna(v) else CONTENT_GROUP.get(v, CONTENT_GROUP_FALLBACK)
    )
    out["faculty_group"] = out["faculty"].map(faculty_group)

    # keep the new columns where the Thai ones were; the raw free-text faculty
    # answer is dropped so the file holds no Thai at all (data.csv still has it)
    order = []
    for col in out.columns:
        if col == "faculty":
            order.append("faculty_group")
        elif col in ("faculty_group", "content_group"):
            continue
        else:
            order.append(col)
            if col == "content_type":
                order.append("content_group")
    return out[order]


def to_codes(df: pd.DataFrame) -> pd.DataFrame:
    """Numeric-coded copy for SPSS: no text cells, blanks stay blank."""
    out = df.copy()
    for col, labels in CODE_ORDER.items():
        codes = {label: i for i, label in enumerate(labels, start=1)}
        out[col] = out[col].map(codes)
    return out


def codebook() -> pd.DataFrame:
    """variable / code / label table describing every coded column."""
    rows = [
        {"variable": col, "code": i, "label": label}
        for col, labels in CODE_ORDER.items()
        for i, label in enumerate(labels, start=1)
    ]
    return pd.DataFrame(rows)


def spss_value_labels() -> str:
    """VALUE LABELS syntax so SPSS shows names instead of the numbers."""
    blocks = []
    for col, labels in CODE_ORDER.items():
        lines = [
            f"  {i} '{label.replace(chr(39), chr(39) * 2)}'"
            for i, label in enumerate(labels, start=1)
        ]
        blocks.append(f"VALUE LABELS {col}\n" + "\n".join(lines) + ".")
    return "\n\n".join(blocks) + "\n"

## 1.4 ทำความสะอาดและคิดคะแนน DASS-21

`clean()` ทำตามลำดับนี้

1. **ตั้งชื่อข้อ DASS-21** คอลัมน์คำถามภาษาไทยที่ขึ้นต้นด้วยเลขข้อ กลายเป็น `dass_1` ถึง `dass_21`
2. **ข้อดักความใส่ใจ** ได้ 1 เมื่อผู้ตอบเลือกตัวเลือกที่ข้อความสั่งไว้ ("ไม่ค่อยตรงกับฉัน") ไม่เช่นนั้นได้ 0
3. **แปลงข้อความเป็นตัวเลข** ดึงตัวเลขทุกตัวในช่องมาเฉลี่ยกัน ช่วงอย่าง "10-12" จึงกลายเป็น 11
   และ "2,94" ถูกอ่านเป็น 2.94 เพราะจุลภาคถูกใช้แทนจุดทศนิยม
4. **ตัดค่าที่เป็นไปไม่ได้** ค่าที่อยู่นอกช่วงใน `RANGES` (เช่น นอน 210 ชั่วโมง) กลายเป็นค่าว่าง แทนที่จะปล่อยให้บิดค่าสถิติ
5. **คิดคะแนน** แต่ละมิติคือผลรวม 7 ข้อคูณ 2 ตามเกณฑ์มาตรฐานของ DASS-21 จึงอยู่ในช่วง 0 ถึง 42
   ผู้ตอบที่ข้อใดข้อหนึ่งในมิติว่าง จะไม่มีคะแนนมิตินั้น
6. **ตัดคอลัมน์ที่ไม่ใช้วิเคราะห์** ได้แก่ ประทับเวลา ข้อความยินยอม และคอลัมน์ว่างท้ายไฟล์
7. **แปลงคำตอบแบบตัวเลือกเป็นภาษาอังกฤษ** ด้วย `to_english()` จากหัวข้อ 1.3

In [7]:
NUMBER = re.compile(r"\d+(?:\.\d+)?")

TIMESTAMP = "ประทับเวลา"

ATTENTION = "attention_check"

# column -> (low, high) accepted range; anything else becomes NaN
RANGES = {
    "age": (15, 60),
    "year_of_study": (1, 8),
    "social_media_hours": (0, 24),
    "news_days": (0, 7),
    "sleep_hours": (0, 24),
    "exercise_days": (0, 7),
    "sleep_quality": (1, 10),
    "study_pressure": (1, 10),
    "GPA": (0, 4),
    "financial_sufficiency": (1, 10),
}

CATEGORICAL = ["gender", "faculty", "main_platform", "content_type"]

# the attention-check item asks the respondent to pick option 3
ATTENTION_ANSWER = "ไม่ค่อยตรงกับฉัน"


def to_number(value) -> float:
    """Mean of every number in the cell; NaN when the cell holds none."""
    if pd.isna(value):
        return np.nan
    # "2,94" is a comma used as a decimal point, not two numbers
    text = re.sub(r"(?<=\d),(?=\d)", ".", str(value))
    found = NUMBER.findall(text)
    if not found:
        return np.nan
    return float(np.mean([float(n) for n in found]))


def clip_to_range(s: pd.Series, low: float, high: float) -> pd.Series:
    """NaN out values that cannot be real answers (typos such as 210 hours)."""
    return s.where(s.between(low, high))


def clean(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy().rename(columns=RENAME)

    # the 21 DASS items are the numbered Thai questions; give them short names
    dass = [c for c in out.columns if re.match(r"^\d+\.\s", str(c))]
    out = out.rename(columns={c: f"dass_{c.split('.', 1)[0]}" for c in dass})
    dass = [f"dass_{c.split('.', 1)[0]}" for c in dass]

    # attention check -> 1 when the respondent picked the requested option
    check = [c for c in out.columns if ATTENTION_ANSWER in str(c)]
    if check:
        out[ATTENTION] = (
            out[check[0]].astype(str).str.strip() == ATTENTION_ANSWER
        ).astype(int)
        out = out.drop(columns=check)

    for col in list(RANGES) + dass:
        if col not in out.columns:
            continue
        out[col] = out[col].map(to_number)

    for col, (low, high) in RANGES.items():
        if col in out.columns:
            out[col] = clip_to_range(out[col], low, high)
    for col in dass:
        out[col] = clip_to_range(out[col], 0, 3)

    # DASS-21 subscale and total scores (x2 is the standard scoring)
    for name, items in DASS_ITEMS.items():
        cols = [f"dass_{i}" for i in items if f"dass_{i}" in out.columns]
        out[name] = out[cols].sum(axis=1, min_count=len(cols)) * 2
    out["dass_total"] = out[list(DASS_ITEMS)].sum(axis=1, min_count=len(DASS_ITEMS))

    # free-text metadata that carries no analysable value
    drop = [c for c in out.columns if c == TIMESTAMP or str(c).startswith("คอลัมน์")]
    drop += [c for c in out.columns if "ยินยอม" in str(c) or "ข้อมูลที่สอบถาม" in str(c)]
    out = out.drop(columns=drop)

    for col in CATEGORICAL:
        if col in out.columns:
            out[col] = (
                out[col]
                .astype(str)
                .str.strip()
                .replace({"": np.nan, "-": np.nan, "_": np.nan})
            )

    return to_english(out)


def report_dropped(raw: pd.DataFrame, out: pd.DataFrame) -> None:
    """Show which cells the cleaner could not turn into a usable number."""
    rows = []
    for col in RANGES:
        if col not in raw.columns:
            continue
        lost = raw[col][out[col].isna() & raw[col].notna()]
        for i, v in lost.items():
            rows.append({"row": i, "column": col, "raw_value": v})
    if rows:
        display(pd.DataFrame(rows))
    else:
        print("no values dropped")

ตัวอย่างการแปลงของ `to_number()` กับรูปแบบคำตอบที่พบในแบบสอบถาม (ตัวอย่างนี้พิมพ์ขึ้นเอง ไม่ใช่คำตอบของผู้ตอบ)

In [8]:
examples = ["8.5 ชั่วโมง", "6-7 ชม.", "10-12", "ปี 2", "2,94", "-"]
display(pd.DataFrame({"answer": examples, "to_number": [to_number(v) for v in examples]}))

,answer,to_number
0,8.5 ชั่วโมง,8.50
1,6-7 ชม.,6.50
2,10-12,11.00
3,ปี 2,2.00
4,"2,94",2.94
5,-,NaN


## 1.5 รันการทำความสะอาดและบันทึกไฟล์

เขียน `data/data_clean.csv` ไว้ให้ notebook ถัดไปทุกไฟล์อ่าน และเขียนไฟล์สำหรับ SPSS ชุดเดียวกันโดยแทนหมวดหมู่ด้วยรหัส 1 ถึง k
พร้อม codebook และคำสั่ง `VALUE LABELS`

In [9]:
cleaned = clean(raw)
cleaned.to_csv(CLEAN_PATH, index=False)
print(f"cleaned: {cleaned.shape[0]} rows x {cleaned.shape[1]} columns -> {CLEAN_PATH}")

# SPSS reads numbers, not words: same rows with every category coded 1..k
spss = to_codes(cleaned)
spss.to_csv(SPSS_PATH, index=False)
codebook().to_csv(CODEBOOK_PATH, index=False)
SPSS_SYNTAX_PATH.write_text(spss_value_labels(), encoding="utf-8")
print(
    f"spss: {spss.shape[0]} rows x {spss.shape[1]} columns "
    f"-> {SPSS_PATH} (codes in {CODEBOOK_PATH}, syntax in {SPSS_SYNTAX_PATH})"
)

left = [
    c
    for c in cleaned.columns
    if cleaned[c].astype(str).str.contains(r"[฀-๿]", regex=True).any()
]
if left:
    print(f"still Thai: {left}")

cleaned: 121 rows x 41 columns -> data/data_clean.csv
spss: 121 rows x 41 columns -> data/data_clean_spss.csv (codes in data/spss_codebook.csv, syntax in data/spss_value_labels.sps)


ช่องที่แปลงเป็นตัวเลขที่ใช้ได้ไม่สำเร็จ ถูกทิ้งเป็นค่าว่าง ตารางนี้แสดงว่าทิ้งอะไรไปบ้าง
ตามด้วยค่าสรุปของคอลัมน์ตัวเลขหลังทำความสะอาด

In [10]:
report_dropped(raw, cleaned)
display(cleaned[list(RANGES)].describe().T.round(2))

,row,column,raw_value
0,32,sleep_hours,210
1,23,GPA,-
2,44,GPA,-
3,62,GPA,-
4,91,GPA,-


,count,mean,std,min,25%,50%,75%,max
age,121.0,19.33,0.79,17.0,19.00,19.00,20.00,23.0
year_of_study,121.0,1.88,0.41,1.0,2.00,2.00,2.00,3.0
social_media_hours,121.0,7.79,2.69,3.0,6.00,7.20,9.50,15.5
news_days,121.0,3.50,1.51,1.0,3.00,4.00,4.00,7.0
sleep_hours,120.0,6.20,1.51,2.0,5.00,6.00,7.00,11.0
exercise_days,121.0,2.12,1.82,0.0,0.00,2.00,3.00,7.0
sleep_quality,121.0,5.61,2.08,1.0,4.00,6.00,7.00,10.0
study_pressure,121.0,6.07,2.26,1.0,5.00,7.00,7.00,10.0
GPA,117.0,3.18,0.50,2.0,2.89,3.25,3.54,4.0
financial_sufficiency,121.0,6.48,2.12,1.0,5.00,7.00,8.00,10.0


## 1.6 กลุ่มตัวอย่างที่ใช้วิเคราะห์

เค้าโครงกำหนดให้ตัดผู้ที่ตอบข้อดักความใส่ใจผิดออก การวิเคราะห์หลักทุกข้อจึงใช้ `df` (ผู้ผ่านข้อดัก)
ส่วน `survey` (ผู้ตอบทั้งหมด) ใช้เฉพาะการตรวจความทนทานของผลใน `06_q6_q7_regression.ipynb` กราฟใน `09_3d_what_if.ipynb`
และไฟล์ส่งออกใน `10_export_website.ipynb` ทุกไฟล์อ่านสองกลุ่มนี้ด้วย `load_data()` และ `passed_attention()` จาก `common.py`

In [11]:
survey = load_data()  # every response, cleaned
df = passed_attention(survey)  # the sample every research question uses
print(
    f"{len(survey)} responses -> {len(df)} passed the attention check "
    f"({len(survey) - len(df)} dropped)"
)
print(f"gender: {df['gender'].value_counts().to_dict()}")
print(f"content group: {df['content_group'].value_counts().to_dict()}")
display(df[list(DIMENSIONS) + ["dass_total"]].describe().T.round(2))

121 responses -> 107 passed the attention check (14 dropped)
gender: {'Female': 59, 'Male': 48}
content group: {'Entertainment': 58, 'Other': 28, "Others' daily life": 13, 'News': 8}


,count,mean,std,min,25%,50%,75%,max
dass_depression,107.0,9.42,8.13,0.0,4.0,6.0,15.0,36.0
dass_anxiety,107.0,11.40,7.47,0.0,6.0,10.0,16.0,38.0
dass_stress,107.0,11.89,7.78,0.0,6.0,10.0,16.0,34.0
dass_total,107.0,32.71,19.99,0.0,18.0,26.0,46.0,92.0
